In [20]:
# Ep5 Code Lab: Free APIs & Integration
# Google Colab version

!pip -q install requests

import requests
import json
from google.colab import userdata

# 1. CONFIG
BASE_URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL_NAME = "openai/gpt-oss-120b"

# Read API key from Colab Secrets
# In Colab: click the key icon (Secrets) and create GROQ_API_KEY
API_KEY = userdata.get("GROQ_API_KEY")

if not API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets.")

print("Connected. Ready.")

# 2. QUICK TEST — raw request
def raw_call(prompt, model=MODEL_NAME):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    response = requests.post(BASE_URL, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"]

print(raw_call("Say hello in one short sentence."))

Connected. Ready.
Hello!


In [16]:
# 3. THE 5 UNIVERSAL API COMPONENTS

# 1. BASE URL — address of the model server
BASE_URL = "https://api.groq.com/openai/v1/chat/completions"

# 2. AUTH KEY — proves you are authorised
API_KEY = userdata.get("GROQ_API_KEY")

# 3. AUTH HEADER
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# 4. MODEL STRING + 5. MESSAGES LIST
payload = {
    "model": "openai/gpt-oss-120b",
    "messages": [
        {"role": "user", "content": "Hello"}
    ]
}

# Send and parse
response = requests.post(BASE_URL, headers=headers, json=payload)
response.raise_for_status()
data = response.json()

print(data["choices"][0]["message"]["content"])

Hello! How can I assist you today?


In [21]:
# 4. THE MESSAGES LIST IN DEPTH

full_call = requests.post(
    BASE_URL,
    headers=headers,
    json={
        "model": "openai/gpt-oss-120b",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is the capital of France?"}
        ]
    }
)

#full_call.raise_for_status()
#data = full_call.json()
print("Status:", full_call.status_code)
print("Response:", full_call.text)

if full_call.ok:
    data = full_call.json()
    print("Answer:", data["choices"][0]["message"]["content"])
else:
    print("Request failed. Check the response above.")

print("Answer:", data["choices"][0]["message"]["content"])
print("Tokens used:", data.get("usage", {}).get("total_tokens", "Not returned"))

Status: 200
Response: {"id":"chatcmpl-774ba96d-6b61-4ff8-a226-d78571894e1f","object":"chat.completion","created":1792950345,"model":"openai/gpt-oss-120b","choices":[{"index":0,"message":{"role":"assistant","content":"The capital of France is **Paris**.","reasoning":"The user asks simple factual question: capital of France. Answer: Paris."},"logprobs":null,"finish_reason":"stop"}],"usage":{"queue_time":0.243590475,"prompt_tokens":87,"prompt_time":0.003541117,"completion_tokens":34,"completion_time":0.070431602,"total_tokens":121,"total_time":0.073972719,"completion_tokens_details":{"reasoning_tokens":16}},"usage_breakdown":null,"system_fingerprint":"fp_a655021f7d","x_groq":{"id":"req_01m5t2p7mde569n56jvgzsjg8q","seed":1342289797},"service_tier":"on_demand"}

Answer: The capital of France is **Paris**.
Answer: The capital of France is **Paris**.
Tokens used: 121


In [22]:
# 5. WRAPPER CLASS

class GroqClient:
    def __init__(self, api_key, base_url=BASE_URL, default_model=MODEL_NAME):
        self.api_key = api_key
        self.base_url = base_url
        self.default_model = default_model

    def chat(self, messages, model=None, temperature=0):
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        payload = {
            "model": model or self.default_model,
            "messages": messages,
            "temperature": temperature
        }

        response = requests.post(self.base_url, headers=headers, json=payload)
        response.raise_for_status()
        data = response.json()

        return data["choices"][0]["message"]["content"]

client = GroqClient(api_key=API_KEY)

In [25]:
# 6. TESTING THE WRAPPER

# Single call
answer = client.chat([
    {"role": "user", "content": "What is 900 * 29?"}
])
print("Single call:", answer)

# Multi-turn conversation
conversation = [
    {"role": "user", "content": "My name is Mochoge."},
    {"role": "assistant", "content": "Hello Mochoge!"},
    {"role": "user", "content": "What is my name?"}
]

print()
print("Multi-turn:", client.chat(conversation))

Single call: 900 × 29 = 26,100.

Multi-turn: Your name is Mochoge.


In [26]:
# 7. FIVE EPISODES INTO ONE PIPELINE

user_name = "Mochoge"

system_instruction = f"""
ROLE: You are a digital twin for {user_name}.
KNOWLEDGE: Activity data from Ep1, embeddings from Ep2, attention from Ep3, prompting strategy from Ep4.
CONSTRAINTS: Only recommend based on user data.
TONE: Direct and analytical.
"""

twin_messages = [
    {"role": "system", "content": system_instruction},
    {"role": "user", "content": "What should I do next?"}
]

response = client.chat(twin_messages)
print(response)

I can only generate a recommendation that’s grounded in the activity, embedding, and attention data I have for you. At the moment I don’t have any of that information in this session, so I’m unable to suggest a concrete next step.

Please provide the most recent details you have—e.g., the last few actions you took, the current focus of your embeddings (goals, topics, or projects), and any attention‑weight signals (what you’ve been prioritizing or flagging). Once I have that data, I’ll give you a direct, data‑driven recommendation for what to do next.


In [27]:
# 8. STRETCH: TWO-TURN CONVERSATION

two_turn_messages = [
    {"role": "system", "content": system_instruction},
    {"role": "user", "content": "What should I do next?"},
    {"role": "assistant", "content": client.chat(twin_messages)},
    {"role": "user", "content": "Give me the top 3 priorities only."}
]

print(client.chat(two_turn_messages))

I can’t generate a data‑driven priority list without any of your activity, embedding, or attention information. Please share at least one of the following so I can produce a concrete “top‑3” recommendation:

- Recent activity logs (tasks completed, time spent, physiological metrics, etc.)  
- Your current embedding profile (goals, interests, self‑assessment results)  
- Attention focus data (domains or projects you’ve been concentrating on lately)

Once I have that input, I’ll analyze it and give you the top three priorities.
